In [ ]:
+-# ============================================
# NOTEBOOK 2 — EXPERIMENTAL PIPELINES
# Cell 1: Define the frozen 30-task benchmark
# ============================================

SELECTED_TASKS = {
    "easy": [
        53, 23, 45, 27, 15,
        16, 98, 35, 13, 83
    ],

    "medium": [
        50, 58, 89, 26, 154,
        140, 111, 93, 56, 31
    ],

    "hard": [
        7, 25, 132, 54, 20,
        6, 41, 126, 43, 99
    ]
}

# Check the selection
print("Frozen benchmark created.")
print(f"Easy   : {len(SELECTED_TASKS['easy'])} tasks")
print(f"Medium : {len(SELECTED_TASKS['medium'])} tasks")
print(f"Hard   : {len(SELECTED_TASKS['hard'])} tasks")
print(f"Total  : {sum(len(v) for v in SELECTED_TASKS.values())} tasks")

Frozen benchmark created.
Easy   : 10 tasks
Medium : 10 tasks
Hard   : 10 tasks
Total  : 30 tasks


In [ ]:
# ============================================
# Cell 2: Load the selected HumanEval records
# ============================================

from datasets import load_dataset

# Load HumanEval test split
humaneval = load_dataset("openai/openai_humaneval", split="test")

# Create a lookup dictionary using the numeric HumanEval ID
task_lookup = {}

for task in humaneval:
    numeric_id = int(task["task_id"].split("/")[-1])
    task_lookup[numeric_id] = task

# Retrieve the 30 frozen tasks
core_tasks = []

for difficulty, task_ids in SELECTED_TASKS.items():
    for task_id in task_ids:
        task = task_lookup[task_id].copy()
        task["difficulty"] = difficulty
        core_tasks.append(task)

print("Selected HumanEval records loaded successfully.")
print(f"Total tasks: {len(core_tasks)}")

Selected HumanEval records loaded successfully.
Total tasks: 30


In [ ]:
# ============================================
# Cell 3: Verify the 30-task benchmark
# ============================================

required_fields = [
    "task_id",
    "prompt",
    "test",
    "entry_point",
    "difficulty"
]

all_valid = True

for task in core_tasks:
    missing = [field for field in required_fields if field not in task]

    if missing:
        print(f"Missing fields in {task['task_id']}: {missing}")
        all_valid = False

if all_valid:
    print("✓ All 30 tasks contain the required fields.")
else:
    print("✗ Some tasks are missing required fields.")

✓ All 30 tasks contain the required fields.


In [ ]:
# ============================================
# Cell 4: Verify Hugging Face authentication
# ============================================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("✓ Hugging Face token is available.")
else:
    print("✗ HF_TOKEN not found.")
    print("Add HF_TOKEN through Colab Secrets (🔑) before continuing.")

✓ Hugging Face token is available.


In [ ]:
# ============================================
# Cell 5: Lock the experimental model
# ============================================

MODEL = "meta-llama/Llama-3.1-8B-Instruct"

print("Experimental model locked:")
print(MODEL)

Experimental model locked:
meta-llama/Llama-3.1-8B-Instruct


In [ ]:
# ============================================
# Cell 6: Create Hugging Face Inference Client
# ============================================

from huggingface_hub import InferenceClient

client = InferenceClient(
    model=MODEL,
    token=HF_TOKEN
)

print("✓ Hugging Face InferenceClient created successfully.")
print(f"✓ Model: {MODEL}")

✓ Hugging Face InferenceClient created successfully.
✓ Model: meta-llama/Llama-3.1-8B-Instruct


In [ ]:
# ============================================
# Cell 4: Load and clean Hugging Face token
# ============================================

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    HF_TOKEN = HF_TOKEN.strip()
    print("✓ Hugging Face token loaded successfully.")
    print("✓ Token whitespace cleaned.")
else:
    print("✗ HF_TOKEN not found.")

✓ Hugging Face token loaded successfully.
✓ Token whitespace cleaned.


In [ ]:
# ============================================
# Cell 6: Recreate Hugging Face client
# ============================================

from huggingface_hub import InferenceClient

client = InferenceClient(
    model=MODEL,
    token=HF_TOKEN
)

print("✓ Hugging Face InferenceClient created successfully.")
print(f"✓ Model: {MODEL}")

✓ Hugging Face InferenceClient created successfully.
✓ Model: meta-llama/Llama-3.1-8B-Instruct


In [ ]:
# ============================================
# Cell 7: Test the locked LLM
# ============================================

response = client.chat_completion(
    messages=[
        {
            "role": "user",
            "content": "Write a Python function that adds two numbers. Return only the code."
        }
    ],
    temperature=0.01,
    max_tokens=256
)

print("✓ Model response received:\n")
print(response.choices[0].message.content)

✓ Model response received:

```python
def add_numbers(a, b):
    return a + b
```


In [ ]:
# ============================================
# Cell 8: Reusable LLM Call Function
# ============================================

import time

def call_llm(system_prompt, user_prompt, max_tokens=1024):
    """
    Send one request to the locked Hugging Face model.

    Returns:
        content      -> model's text response
        latency_sec  -> API response time
        tokens       -> token count if provided by API,
                        otherwise a consistent estimate
    """

    start_time = time.perf_counter()

    response = client.chat_completion(
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.01,
        max_tokens=max_tokens
    )

    end_time = time.perf_counter()

    content = response.choices[0].message.content
    latency_sec = end_time - start_time

    # Try to obtain actual token usage
    usage = getattr(response, "usage", None)

    if usage is not None and getattr(usage, "total_tokens", None) is not None:
        tokens = usage.total_tokens
        token_source = "api"
    else:
        # Consistent fallback estimate
        tokens = len(content.split()) * 1.3
        token_source = "estimated"

    return content, latency_sec, tokens, token_source


print("✓ Reusable LLM call function created.")

✓ Reusable LLM call function created.


In [ ]:
# ============================================
# Cell 9: Pipeline A — Single Agent
# ============================================

SINGLE_AGENT_SYSTEM_PROMPT = """
You are an expert Python developer.

Solve the given HumanEval programming problem.

Requirements:
1. Understand the problem carefully.
2. Implement the required function.
3. Return only executable Python code.
4. Do not include explanations.
5. Do not include Markdown code fences.
6. Keep the required function name and signature unchanged.
"""

def single_agent_pipeline(task):
    """
    Pipeline A:
    HumanEval task → Single LLM call → Generated code
    """

    task_id = task["task_id"]
    prompt = task["prompt"]

    code, latency, tokens, token_source = call_llm(
        system_prompt=SINGLE_AGENT_SYSTEM_PROMPT,
        user_prompt=prompt,
        max_tokens=1024
    )

    result = {
        "task_id": task_id,
        "difficulty": task["difficulty"],
        "generated_code": code,
        "latency_sec": latency,
        "tokens": tokens,
        "token_source": token_source
    }

    return result


print("✓ Pipeline A — Single Agent defined successfully.")

✓ Pipeline A — Single Agent defined successfully.


In [ ]:
# ============================================
# Cell 10: Locate core_tasks.json
# ============================================

import os

matches = []

for root, dirs, files in os.walk("/content"):
    if "core_tasks.json" in files:
        matches.append(os.path.join(root, "core_tasks.json"))

print("Files found:")

if matches:
    for path in matches:
        print("✓", path)
else:
    print("✗ core_tasks.json was not found anywhere in /content")

Files found:
✗ core_tasks.json was not found anywhere in /content


In [ ]:
# ============================================
# Cell 10: Load HumanEval Dataset
# ============================================

from datasets import load_dataset

print("Loading HumanEval dataset...")

ds = load_dataset(
    "openai/openai_humaneval",
    split="test"
)

print("✓ HumanEval dataset loaded")
print("Number of tasks:", len(ds))
print("Columns:", ds.column_names)

Loading HumanEval dataset...
✓ HumanEval dataset loaded
Number of tasks: 164
Columns: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point']


In [ ]:
# ============================================
# Cell 11: Verify HumanEval
# ============================================

print("First task:")
print("Task ID:", ds[0]["task_id"])
print("\nPrompt:")
print(ds[0]["prompt"][:500])
print("\nTest available:", bool(ds[0]["test"]))

First task:
Task ID: HumanEval/0

Prompt:
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


Test available: True


In [ ]:
# ============================================
# Cell 12: Create Tasks Folder
# ============================================

import os

os.makedirs("tasks", exist_ok=True)

print("✓ tasks/ folder ready")

✓ tasks/ folder ready


In [ ]:
# ============================================
# Cell 13: Save Full HumanEval Dataset
# ============================================

import json

all_tasks = []

for item in ds:
    all_tasks.append({
        "task_id": item["task_id"],
        "prompt": item["prompt"],
        "test": item["test"],
        "entry_point": item["entry_point"]
    })

with open("tasks/humaneval_full.json", "w") as f:
    json.dump(all_tasks, f, indent=2)

print("✓ Full HumanEval dataset saved")
print("Tasks:", len(all_tasks))
print("File: tasks/humaneval_full.json")

✓ Full HumanEval dataset saved
Tasks: 164
File: tasks/humaneval_full.json


In [ ]:
# ============================================
# Cell 13.5: Import pandas
# ============================================

import pandas as pd

print("✓ pandas imported successfully.")

✓ pandas imported successfully.


In [ ]:
# ============================================
# Cell 14: Create Final Core Task Set
# ============================================

import json

# Your finalized 30 HumanEval tasks
task_difficulty = {
    "HumanEval/53": "easy",
    "HumanEval/23": "easy",
    "HumanEval/45": "easy",
    "HumanEval/27": "easy",
    "HumanEval/15": "easy",
    "HumanEval/16": "easy",
    "HumanEval/98": "easy",
    "HumanEval/35": "easy",
    "HumanEval/13": "easy",
    "HumanEval/83": "easy",

    "HumanEval/50": "medium",
    "HumanEval/58": "medium",
    "HumanEval/89": "medium",
    "HumanEval/26": "medium",
    "HumanEval/154": "medium",
    "HumanEval/140": "medium",
    "HumanEval/111": "medium",
    "HumanEval/93": "medium",
    "HumanEval/56": "medium",
    "HumanEval/31": "medium",

    "HumanEval/7": "hard",
    "HumanEval/25": "hard",
    "HumanEval/132": "hard",
    "HumanEval/54": "hard",
    "HumanEval/20": "hard",
    "HumanEval/6": "hard",
    "HumanEval/41": "hard",
    "HumanEval/126": "hard",
    "HumanEval/43": "hard",
    "HumanEval/99": "hard"
}

# Load the complete HumanEval dataset
with open("tasks/humaneval_full.json", "r") as f:
    all_tasks = json.load(f)

# Create lookup dictionary
task_lookup = {
    task["task_id"]: task
    for task in all_tasks
}

# Build final core task list
core_tasks = []

for task_id, difficulty in task_difficulty.items():

    if task_id not in task_lookup:
        raise ValueError(
            f"{task_id} was not found in HumanEval dataset."
        )

    task = task_lookup[task_id].copy()
    task["difficulty"] = difficulty

    core_tasks.append(task)

print("✓ Final core task set created")
print("Total tasks:", len(core_tasks))

✓ Final core task set created
Total tasks: 30


In [ ]:
# ============================================
# Cell 15: Verify Difficulty Distribution
# ============================================

from collections import Counter

difficulty_counts = Counter(
    task["difficulty"]
    for task in core_tasks
)

print("Difficulty distribution:")
print("Easy  :", difficulty_counts["easy"])
print("Medium:", difficulty_counts["medium"])
print("Hard  :", difficulty_counts["hard"])
print("Total :", len(core_tasks))

Difficulty distribution:
Easy  : 10
Medium: 10
Hard  : 10
Total : 30


In [ ]:
# ============================================
# Cell 16: Save core_tasks.json
# ============================================

with open("tasks/core_tasks.json", "w") as f:
    json.dump(core_tasks, f, indent=2)

print("✓ core_tasks.json saved successfully")
print("✓ 30 tasks saved")
print("✓ Location: tasks/core_tasks.json")

✓ core_tasks.json saved successfully
✓ 30 tasks saved
✓ Location: tasks/core_tasks.json


In [ ]:
# ============================================
# Cell 17: Final Verification
# ============================================

with open("tasks/core_tasks.json", "r") as f:
    core_tasks = json.load(f)

print("✓ core_tasks.json loaded successfully")
print("Number of tasks:", len(core_tasks))

print("\nSelected tasks:")

for task in core_tasks:
    print(
        f"{task['task_id']:15} | "
        f"{task['difficulty']:6}"
    )

✓ core_tasks.json loaded successfully
Number of tasks: 30

Selected tasks:
HumanEval/53    | easy  
HumanEval/23    | easy  
HumanEval/45    | easy  
HumanEval/27    | easy  
HumanEval/15    | easy  
HumanEval/16    | easy  
HumanEval/98    | easy  
HumanEval/35    | easy  
HumanEval/13    | easy  
HumanEval/83    | easy  
HumanEval/50    | medium
HumanEval/58    | medium
HumanEval/89    | medium
HumanEval/26    | medium
HumanEval/154   | medium
HumanEval/140   | medium
HumanEval/111   | medium
HumanEval/93    | medium
HumanEval/56    | medium
HumanEval/31    | medium
HumanEval/7     | hard  
HumanEval/25    | hard  
HumanEval/132   | hard  
HumanEval/54    | hard  
HumanEval/20    | hard  
HumanEval/6     | hard  
HumanEval/41    | hard  
HumanEval/126   | hard  
HumanEval/43    | hard  
HumanEval/99    | hard  


In [ ]:
test_task = next(
    task for task in core_tasks
    if task["task_id"] == "HumanEval/53"
)

print(test_task["task_id"])
print(test_task["difficulty"])

HumanEval/53
easy


In [ ]:
# ============================================
# Cell 18: Pipeline A — Smoke Test
# ============================================

test_task = next(
    task for task in core_tasks
    if task["task_id"] == "HumanEval/53"
)

print("Testing Pipeline A")
print("Task ID:", test_task["task_id"])
print("Difficulty:", test_task["difficulty"])
print("-" * 60)

pipeline_a_result = single_agent_pipeline(test_task)

print("\n✓ Pipeline A smoke test completed")
print("Task ID:", pipeline_a_result["task_id"])
print("Latency:", round(pipeline_a_result["latency_sec"], 2), "seconds")
print("Tokens:", pipeline_a_result["tokens"])
print("Token source:", pipeline_a_result["token_source"])

print("\nGenerated code:")
print(pipeline_a_result["generated_code"])

Testing Pipeline A
Task ID: HumanEval/53
Difficulty: easy
------------------------------------------------------------

✓ Pipeline A smoke test completed
Task ID: HumanEval/53
Latency: 1.14 seconds
Tokens: 145
Token source: api

Generated code:
def add(x: int, y: int):
    return x + y


In [ ]:
# ============================================
# Final Cell: Pipeline A Smoke Test — 3 Tasks
# ============================================

smoke_task_ids = [
    "HumanEval/53",   # Easy
    "HumanEval/50",   # Medium
    "HumanEval/7"     # Hard
]

pipeline_a_smoke_results = []

for task_id in smoke_task_ids:

    task = next(
        task for task in core_tasks
        if task["task_id"] == task_id
    )

    result = single_agent_pipeline(task)

    pipeline_a_smoke_results.append({
        "task_id": result["task_id"],
        "difficulty": result["difficulty"],
        "latency_sec": result["latency_sec"],
        "tokens": result["tokens"],
        "token_source": result["token_source"],
        "generated_code": result["generated_code"]
    })

    print("=" * 60)
    print("Task:", result["task_id"])
    print("Difficulty:", result["difficulty"])
    print("Latency:", round(result["latency_sec"], 2), "sec")
    print("Tokens:", result["tokens"])
    print("Code:")
    print(result["generated_code"])

print("\n✓ Pipeline A 3-task smoke test completed.")

Task: HumanEval/53
Difficulty: easy
Latency: 1.07 sec
Tokens: 145
Code:
def add(x: int, y: int):
    return x + y
Task: HumanEval/50
Difficulty: medium
Latency: 2.96 sec
Tokens: 225
Code:
def decode_shift(s: str):
    """
    takes as input string encoded with encode_shift function. Returns decoded string.
    """
    return "".join([chr(((ord(ch) - 5 - ord("a")) % 26) + ord("a")) for ch in s])
Task: HumanEval/7
Difficulty: hard
Latency: 1.81 sec
Tokens: 201
Code:
def filter_by_substring(strings: List[str], substring: str) -> List[str]:
    return [s for s in strings if substring in s]

✓ Pipeline A 3-task smoke test completed.


In [ ]:
# ============================================
# Full Pipeline A — 30 Core Tasks
# ============================================

import json
import time

pipeline_a_results = []

print("Starting Pipeline A on 30 core tasks...")
print("=" * 60)

for i, task in enumerate(core_tasks, start=1):

    print(
        f"[{i}/30] {task['task_id']} "
        f"({task['difficulty']})"
    )

    try:
        result = single_agent_pipeline(task)

        pipeline_a_results.append(result)

        print(
            f"  ✓ Latency: "
            f"{result['latency_sec']:.2f}s | "
            f"Tokens: {result['tokens']}"
        )

    except Exception as e:

        print(f"  ✗ ERROR: {e}")

        pipeline_a_results.append({
            "task_id": task["task_id"],
            "difficulty": task["difficulty"],
            "generated_code": "",
            "latency_sec": None,
            "tokens": None,
            "token_source": "error",
            "error": str(e)
        })

print("=" * 60)
print(
    "✓ Pipeline A completed:",
    len(pipeline_a_results),
    "tasks"
)

Starting Pipeline A on 30 core tasks...
[1/30] HumanEval/53 (easy)
  ✓ Latency: 1.02s | Tokens: 145
[2/30] HumanEval/23 (easy)
  ✓ Latency: 1.29s | Tokens: 134
[3/30] HumanEval/45 (easy)
  ✓ Latency: 1.36s | Tokens: 140
[4/30] HumanEval/27 (easy)
  ✓ Latency: 1.54s | Tokens: 149
[5/30] HumanEval/15 (easy)
  ✓ Latency: 2.12s | Tokens: 169
[6/30] HumanEval/16 (easy)
  ✓ Latency: 1.80s | Tokens: 166
[7/30] HumanEval/98 (easy)
  ✓ Latency: 3.22s | Tokens: 203
[8/30] HumanEval/35 (easy)
  ✓ Latency: 0.99s | Tokens: 171
[9/30] HumanEval/13 (easy)
  ✓ Latency: 2.08s | Tokens: 181
[10/30] HumanEval/83 (easy)
  ✗ ERROR: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8481e9-33b4f7f71ea56d7b06370347;ceb8a594-ecc0-4394-9ce1-37a99af622a3)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference 

In [ ]:
# ============================================
# Full Pipeline A — 30 Core Tasks
# ============================================

import json
import time

pipeline_a_results = []

print("Starting Pipeline A on 30 core tasks...")
print("=" * 60)

for i, task in enumerate(core_tasks, start=1):

    print(
        f"[{i}/30] {task['task_id']} "
        f"({task['difficulty']})"
    )

    try:
        result = single_agent_pipeline(task)

        pipeline_a_results.append(result)

        print(
            f"  ✓ Latency: "
            f"{result['latency_sec']:.2f}s | "
            f"Tokens: {result['tokens']}"
        )

    except Exception as e:

        print(f"  ✗ ERROR: {e}")

        pipeline_a_results.append({
            "task_id": task["task_id"],
            "difficulty": task["difficulty"],
            "generated_code": "",
            "latency_sec": None,
            "tokens": None,
            "token_source": "error",
            "error": str(e)
        })

print("=" * 60)
print(
    "✓ Pipeline A completed:",
    len(pipeline_a_results),
    "tasks"
)

Starting Pipeline A on 30 core tasks...
[1/30] HumanEval/53 (easy)
  ✓ Latency: 1.34s | Tokens: 145
[2/30] HumanEval/23 (easy)
  ✓ Latency: 1.08s | Tokens: 134
[3/30] HumanEval/45 (easy)
  ✓ Latency: 1.60s | Tokens: 140
[4/30] HumanEval/27 (easy)
  ✓ Latency: 2.11s | Tokens: 157
[5/30] HumanEval/15 (easy)
  ✓ Latency: 1.32s | Tokens: 172
[6/30] HumanEval/16 (easy)
  ✓ Latency: 1.06s | Tokens: 166
[7/30] HumanEval/98 (easy)
  ✓ Latency: 3.92s | Tokens: 203
[8/30] HumanEval/35 (easy)
  ✓ Latency: 1.47s | Tokens: 171
[9/30] HumanEval/13 (easy)
  ✓ Latency: 1.64s | Tokens: 181
[10/30] HumanEval/83 (easy)
  ✗ ERROR: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a848238-37fbdb7145c8b1123a8c5e2c;c4057b9b-1a66-4bd1-b1ba-a3ca9aca3d45)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference 

In [ ]:
# ============================================
# Preserve 9 Successful Pipeline A Results
# ============================================

import json

successful_a = [
    r for r in pipeline_a_results
    if r.get("generated_code")
    and not r.get("error")
]

with open("pipeline_a_smoke_test_results.json", "w") as f:
    json.dump(successful_a, f, indent=2)

print("✓ Smoke-test results preserved")
print("Tasks saved:", len(successful_a))

for r in successful_a:
    print(
        r["task_id"],
        "|",
        r["difficulty"]
    )

✓ Smoke-test results preserved
Tasks saved: 9
HumanEval/53 | easy
HumanEval/23 | easy
HumanEval/45 | easy
HumanEval/27 | easy
HumanEval/15 | easy
HumanEval/16 | easy
HumanEval/98 | easy
HumanEval/35 | easy
HumanEval/13 | easy


In [ ]:
# ============================================
# Cell 10: Recreate frozen benchmark
# ============================================

# Your FINAL frozen 30-task benchmark
SELECTED_TASKS = {
    "easy": [
        53, 23, 45, 27, 15,
        16, 98, 35, 13, 83
    ],

    "medium": [
        50, 58, 89, 26, 154,
        140, 111, 93, 56, 31
    ],

    "hard": [
        7, 25, 132, 54, 20,
        6, 41, 126, 43, 99
    ]
}

# Build lookup from the already downloaded HumanEval dataset
task_lookup = {}

for task in humaneval:
    numeric_id = int(task["task_id"].split("/")[-1])
    task_lookup[numeric_id] = task

# Reconstruct the 30 core task records
core_tasks = []

for difficulty, task_ids in SELECTED_TASKS.items():
    for task_id in task_ids:
        task = task_lookup[task_id].copy()
        task["difficulty"] = difficulty
        core_tasks.append(task)

# Final verification
print("✓ Frozen benchmark recreated successfully.")
print(f"✓ Total tasks: {len(core_tasks)}")

print("\nDifficulty distribution:")
for difficulty in ["easy", "medium", "hard"]:
    count = sum(
        1 for task in core_tasks
        if task["difficulty"] == difficulty
    )
    print(f"  {difficulty.capitalize()}: {count}")

✓ Frozen benchmark recreated successfully.
✓ Total tasks: 30

Difficulty distribution:
  Easy: 10
  Medium: 10
  Hard: 10


In [ ]:
# ============================================
# Cell 11: Recreate reusable LLM call
# ============================================

import time

def call_llm(system_prompt, user_prompt, max_tokens=1024):

    start_time = time.perf_counter()

    response = client.chat_completion(
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0.01,
        max_tokens=max_tokens
    )

    end_time = time.perf_counter()

    content = response.choices[0].message.content
    latency_sec = end_time - start_time

    usage = getattr(response, "usage", None)

    if usage is not None and getattr(usage, "total_tokens", None) is not None:
        tokens = usage.total_tokens
        token_source = "api"
    else:
        tokens = len(content.split()) * 1.3
        token_source = "estimated"

    return content, latency_sec, tokens, token_source


print("✓ call_llm() recreated successfully.")

✓ call_llm() recreated successfully.


In [ ]:
# ============================================
# Cell 12: Recreate Pipeline A
# ============================================

SINGLE_AGENT_SYSTEM_PROMPT = """
You are an expert Python developer.

Solve the given HumanEval programming problem.

Requirements:
1. Understand the problem carefully.
2. Implement the required function.
3. Return only executable Python code.
4. Do not include explanations.
5. Do not include Markdown code fences.
6. Keep the required function name and signature unchanged.
"""

def single_agent_pipeline(task):

    code, latency, tokens, token_source = call_llm(
        system_prompt=SINGLE_AGENT_SYSTEM_PROMPT,
        user_prompt=task["prompt"],
        max_tokens=1024
    )

    return {
        "task_id": task["task_id"],
        "difficulty": task["difficulty"],
        "generated_code": code,
        "latency_sec": latency,
        "tokens": tokens,
        "token_source": token_source
    }


print("✓ Pipeline A recreated successfully.")

✓ Pipeline A recreated successfully.


In [ ]:
# ============================================
# Cell 11: Recreate Hugging Face client
# ============================================

from google.colab import userdata
from huggingface_hub import InferenceClient

# Load token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

HF_TOKEN = HF_TOKEN.strip()

# Lock the same model used in the experiment
MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# Create the client
client = InferenceClient(
    model=MODEL,
    token=HF_TOKEN
)

print("✓ Hugging Face client recreated successfully.")
print(f"✓ Model: {MODEL}")

✓ Hugging Face client recreated successfully.
✓ Model: meta-llama/Llama-3.1-8B-Instruct


In [ ]:
# ============================================
# Cell 12: Verify Hugging Face client
# ============================================

response = client.chat_completion(
    messages=[
        {
            "role": "user",
            "content": "Write a Python function called add(a, b) that returns their sum. Return only code."
        }
    ],
    temperature=0.01,
    max_tokens=256
)

print("✓ Hugging Face client is working.")
print("\nModel response:")
print(response.choices[0].message.content)

✓ Hugging Face client is working.

Model response:
```python
def add(a, b):
    return a + b
```


In [ ]:
# ============================================
# Cell 13: Pipeline A single-task check
# ============================================

test_task = next(
    task for task in core_tasks
    if task["task_id"] == "HumanEval/53"
)

result = single_agent_pipeline(test_task)

print("✓ Pipeline A is working.")
print("Task:", result["task_id"])
print("Difficulty:", result["difficulty"])
print("Latency:", round(result["latency_sec"], 2), "seconds")
print("Tokens:", result["tokens"])

print("\nGenerated code:")
print(result["generated_code"])

✓ Pipeline A is working.
Task: HumanEval/53
Difficulty: easy
Latency: 1.27 seconds
Tokens: 145

Generated code:
def add(x: int, y: int):
    return x + y


In [ ]:
# ============================================
# Cell 14: Pipeline A — 3-task smoke test
# ============================================

SMOKE_TEST_IDS = [
    "HumanEval/53",   # Easy
    "HumanEval/50",   # Medium
    "HumanEval/7"     # Hard
]

pipeline_a_smoke_results = []

print("Pipeline A — 3-task Smoke Test")
print("=" * 60)

for task_id in SMOKE_TEST_IDS:

    task = next(
        task for task in core_tasks
        if task["task_id"] == task_id
    )

    print(f"\n[{task_id}] ({task['difficulty']})")

    try:
        result = single_agent_pipeline(task)

        pipeline_a_smoke_results.append(result)

        print(f"✓ Latency: {result['latency_sec']:.2f}s")
        print(f"✓ Tokens: {result['tokens']}")

        print("\nGenerated code:")
        print(result["generated_code"])

    except Exception as e:
        print(f"✗ ERROR: {e}")

print("\n" + "=" * 60)
print(
    f"✓ Smoke test completed: "
    f"{len(pipeline_a_smoke_results)}/3 tasks"
)

Pipeline A — 3-task Smoke Test

[HumanEval/53] (easy)
✓ Latency: 1.09s
✓ Tokens: 145

Generated code:
def add(x: int, y: int):
    return x + y

[HumanEval/50] (medium)
✓ Latency: 8.06s
✓ Tokens: 288

Generated code:
def encode_shift(s: str):
    """
    returns encoded string by shifting every character by 5 in the alphabet.
    """
    return "".join([chr(((ord(ch) - ord("a") + 5) % 26) + ord("a")) for ch in s.lower() if ch.isalpha()])


def decode_shift(s: str):
    """
    takes as input string encoded with encode_shift function. Returns decoded string.
    """
    return "".join([chr(((ord(ch) - ord("a")) % 26) + ord("a")) for ch in s])

[HumanEval/7] (hard)
✓ Latency: 3.80s
✓ Tokens: 201

Generated code:
def filter_by_substring(strings: List[str], substring: str) -> List[str]:
    return [s for s in strings if substring in s]

✓ Smoke test completed: 3/3 tasks


In [ ]:
print("Model:", MODEL)
print("Provider:", client.provider)
print("Core tasks:", len(core_tasks))

Model: meta-llama/Llama-3.1-8B-Instruct
Provider: None
Core tasks: 30


In [ ]:
# ============================================
# Final Experiment: Hugging Face Authentication
# ============================================

from google.colab import userdata

HF_TOKEN = userdata.get("SecureAgentEvalExp")

if not HF_TOKEN:
    raise ValueError("SecureAgentEvalExp secret not found.")

print("✓ Hugging Face token loaded successfully.")

✓ Hugging Face token loaded successfully.


In [ ]:
from huggingface_hub import InferenceClient

MODEL = "meta-llama/Llama-3.1-8B-Instruct"

client = InferenceClient(
    model=MODEL,
    token=HF_TOKEN
)

print("✓ Hugging Face client initialized.")
print("Model:", MODEL)
print("Provider:", client.provider)

✓ Hugging Face client initialized.
Model: meta-llama/Llama-3.1-8B-Instruct
Provider: None


In [ ]:
# ============================================
# Restore Official HumanEval Dataset
# ============================================

import os
import json
import gzip
import urllib.request

os.makedirs("tasks", exist_ok=True)

URL = "https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz"

print("Downloading official HumanEval dataset...")

urllib.request.urlretrieve(
    URL,
    "tasks/HumanEval.jsonl.gz"
)

all_tasks = []

with gzip.open("tasks/HumanEval.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            all_tasks.append(json.loads(line))

# Save in the format our notebook uses
with open("tasks/humaneval_full.json", "w", encoding="utf-8") as f:
    json.dump(all_tasks, f, indent=2)

print("✓ Full HumanEval dataset restored")
print("Tasks:", len(all_tasks))
print("File: tasks/humaneval_full.json")

✓ Full HumanEval dataset restored
Tasks: 164
File: tasks/humaneval_full.json


In [ ]:
# ============================================
# Restore Exact 30 Core Tasks
# ============================================

CORE_TASK_IDS = [
    # EASY — 10
    "HumanEval/53",
    "HumanEval/23",
    "HumanEval/45",
    "HumanEval/27",
    "HumanEval/15",
    "HumanEval/16",
    "HumanEval/98",
    "HumanEval/35",
    "HumanEval/13",
    "HumanEval/83",

    # MEDIUM — 10
    "HumanEval/50",
    "HumanEval/58",
    "HumanEval/89",
    "HumanEval/26",
    "HumanEval/154",
    "HumanEval/140",
    "HumanEval/111",
    "HumanEval/93",
    "HumanEval/56",
    "HumanEval/31",

    # HARD — 10
    "HumanEval/7",
    "HumanEval/25",
    "HumanEval/132",
    "HumanEval/54",
    "HumanEval/20",
    "HumanEval/6",
    "HumanEval/41",
    "HumanEval/126",
    "HumanEval/43",
    "HumanEval/99"
]

task_lookup = {
    task["task_id"]: task
    for task in all_tasks
}

core_tasks = [
    task_lookup[task_id]
    for task_id in CORE_TASK_IDS
]

print("✓ Core tasks restored")
print("Total:", len(core_tasks))

print("\nTask IDs:")
for task in core_tasks:
    print(task["task_id"])

✓ Core tasks restored
Total: 30

Task IDs:
HumanEval/53
HumanEval/23
HumanEval/45
HumanEval/27
HumanEval/15
HumanEval/16
HumanEval/98
HumanEval/35
HumanEval/13
HumanEval/83
HumanEval/50
HumanEval/58
HumanEval/89
HumanEval/26
HumanEval/154
HumanEval/140
HumanEval/111
HumanEval/93
HumanEval/56
HumanEval/31
HumanEval/7
HumanEval/25
HumanEval/132
HumanEval/54
HumanEval/20
HumanEval/6
HumanEval/41
HumanEval/126
HumanEval/43
HumanEval/99


In [ ]:
# ============================================
# Add Project Difficulty Labels
# ============================================

EASY_TASKS = {
    "HumanEval/53",
    "HumanEval/23",
    "HumanEval/45",
    "HumanEval/27",
    "HumanEval/15",
    "HumanEval/16",
    "HumanEval/98",
    "HumanEval/35",
    "HumanEval/13",
    "HumanEval/83"
}

MEDIUM_TASKS = {
    "HumanEval/50",
    "HumanEval/58",
    "HumanEval/89",
    "HumanEval/26",
    "HumanEval/154",
    "HumanEval/140",
    "HumanEval/111",
    "HumanEval/93",
    "HumanEval/56",
    "HumanEval/31"
}

HARD_TASKS = {
    "HumanEval/7",
    "HumanEval/25",
    "HumanEval/132",
    "HumanEval/54",
    "HumanEval/20",
    "HumanEval/6",
    "HumanEval/41",
    "HumanEval/126",
    "HumanEval/43",
    "HumanEval/99"
}

for task in core_tasks:
    task_id = task["task_id"]

    if task_id in EASY_TASKS:
        task["difficulty"] = "easy"
    elif task_id in MEDIUM_TASKS:
        task["difficulty"] = "medium"
    elif task_id in HARD_TASKS:
        task["difficulty"] = "hard"
    else:
        raise ValueError(f"Unknown task: {task_id}")

print("✓ Difficulty labels added")

from collections import Counter
print("Difficulty distribution:", Counter(
    task["difficulty"] for task in core_tasks
))

✓ Difficulty labels added
Difficulty distribution: Counter({'easy': 10, 'medium': 10, 'hard': 10})


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("SecureAgentEvalExp")

print("Token loaded:", bool(HF_TOKEN))
print("Starts with hf_:", HF_TOKEN.startswith("hf_"))
print("Token length:", len(HF_TOKEN))

Token loaded: True
Starts with hf_: True
Token length: 37


In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

try:
    who = api.whoami()
    print("✓ Hugging Face authentication successful")
    print("Username:", who.get("name"))
except Exception as e:
    print("✗ Hugging Face authentication failed")
    print(type(e).__name__ + ":", e)

✓ Hugging Face authentication successful
Username: Priyanshi047


In [ ]:
from huggingface_hub import InferenceClient

MODEL = "meta-llama/Llama-3.1-8B-Instruct"

client = InferenceClient(
    model=MODEL,
    provider="auto",
    api_key=HF_TOKEN
)

print("✓ Client initialized")
print("Model:", MODEL)
print("Provider:", client.provider)

✓ Client initialized
Model: meta-llama/Llama-3.1-8B-Instruct
Provider: auto


In [ ]:
# ============================================
# FINAL HUGGING FACE CONNECTION TEST
# ============================================

test_response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a Python programming assistant."
        },
        {
            "role": "user",
            "content": "Return only the Python code for a function add(x, y) that returns x + y."
        }
    ],
    max_tokens=100,
    temperature=0.01
)

print("✓ Hugging Face inference successful")
print("\nResponse:")
print(test_response.choices[0].message.content)

if test_response.usage:
    print("\nTokens:", test_response.usage.total_tokens)

✓ Hugging Face inference successful

Response:
def add(x, y):
    return x + y

Tokens: 53


In [ ]:
pipeline_a_results_final = []

print("✓ Ready for final Pipeline A experiment")
print("Tasks:", len(core_tasks))
print("Model:", MODEL)
print("Platform: Hugging Face")

✓ Ready for final Pipeline A experiment
Tasks: 30
Model: meta-llama/Llama-3.1-8B-Instruct
Platform: Hugging Face


In [ ]:
# ============================================
# FINAL PIPELINE A — 30 TASKS
# ============================================

pipeline_a_results_final = []

print("=" * 60)
print("FINAL PIPELINE A — START")
print("Model:", MODEL)
print("Platform: Hugging Face")
print("Tasks:", len(core_tasks))
print("=" * 60)

for i, task in enumerate(core_tasks, start=1):

    print(
        f"\n[{i}/30] "
        f"{task['task_id']} | "
        f"{task['difficulty']}"
    )

    try:
        result = single_agent_pipeline(task)

        pipeline_a_results_final.append(result)

        print(
            f"✓ Completed | "
            f"Latency: {result['latency_sec']:.2f}s | "
            f"Tokens: {result['tokens']}"
        )

    except Exception as e:

        print(f"✗ ERROR: {e}")

        pipeline_a_results_final.append({
            "task_id": task["task_id"],
            "difficulty": task["difficulty"],
            "generated_code": "",
            "latency_sec": None,
            "tokens": None,
            "token_source": "error",
            "error": str(e)
        })

print("\n" + "=" * 60)
print("FINAL PIPELINE A — COMPLETE")
print("Total records:", len(pipeline_a_results_final))
print("=" * 60)

FINAL PIPELINE A — START
Model: meta-llama/Llama-3.1-8B-Instruct
Platform: Hugging Face
Tasks: 30

[1/30] HumanEval/53 | easy
✓ Completed | Latency: 0.88s | Tokens: 144

[2/30] HumanEval/23 | easy
✓ Completed | Latency: 1.05s | Tokens: 133

[3/30] HumanEval/45 | easy
✓ Completed | Latency: 0.76s | Tokens: 139

[4/30] HumanEval/27 | easy
✓ Completed | Latency: 1.08s | Tokens: 147

[5/30] HumanEval/15 | easy
✓ Completed | Latency: 1.96s | Tokens: 168

[6/30] HumanEval/16 | easy
✓ Completed | Latency: 0.97s | Tokens: 165

[7/30] HumanEval/98 | easy
✓ Completed | Latency: 2.16s | Tokens: 202

[8/30] HumanEval/35 | easy
✓ Completed | Latency: 0.76s | Tokens: 170

[9/30] HumanEval/13 | easy
✗ ERROR: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b03ab-04cc95902e3fd14f5232e885;73729829-5ea2-45da-aa70-a92d56c7601d)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted y

In [ ]:
# ============================================
# Check Completed Pipeline A Tasks
# ============================================

completed = [
    r for r in pipeline_a_results_final
    if not r.get("error")
]

failed = [
    r for r in pipeline_a_results_final
    if r.get("error")
]

print("=" * 60)
print("PIPELINE A RUN STATUS")
print("=" * 60)

print("Total records:", len(pipeline_a_results_final))
print("Completed:", len(completed))
print("Failed:", len(failed))

print("\nCompleted tasks:")
for r in completed:
    print(
        r["task_id"],
        "|",
        r["difficulty"],
        "|",
        f"{r['latency_sec']:.2f}s",
        "|",
        r["tokens"],
        "tokens"
    )

print("\nFirst failed task:")
if failed:
    print(failed[0]["task_id"], "|", failed[0]["difficulty"])
    print(failed[0]["error"])

PIPELINE A RUN STATUS
Total records: 30
Completed: 8
Failed: 22

Completed tasks:
HumanEval/53 | easy | 0.88s | 144 tokens
HumanEval/23 | easy | 1.05s | 133 tokens
HumanEval/45 | easy | 0.76s | 139 tokens
HumanEval/27 | easy | 1.08s | 147 tokens
HumanEval/15 | easy | 1.96s | 168 tokens
HumanEval/16 | easy | 0.97s | 165 tokens
HumanEval/98 | easy | 2.16s | 202 tokens
HumanEval/35 | easy | 0.76s | 170 tokens

First failed task:
HumanEval/13 | easy
Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a8b03ab-04cc95902e3fd14f5232e885;73729829-5ea2-45da-aa70-a92d56c7601d)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.


In [ ]:
# ============================================
# SAVE PIPELINE A CHECKPOINT — 8 COMPLETED
# ============================================

import json
import os

pipeline_a_completed = [
    r for r in pipeline_a_results_final
    if not r.get("error")
]

os.makedirs("results", exist_ok=True)

CHECKPOINT_PATH = "results/pipeline_a_checkpoint_8.json"

with open(CHECKPOINT_PATH, "w") as f:
    json.dump(pipeline_a_completed, f, indent=2)

print("✓ Pipeline A checkpoint saved")
print("Completed tasks:", len(pipeline_a_completed))
print("File:", CHECKPOINT_PATH)

✓ Pipeline A checkpoint saved
Completed tasks: 8
File: results/pipeline_a_checkpoint_8.json


In [ ]:
# ============================================
# LOAD EXISTING PIPELINE A CHECKPOINT
# ============================================

import json
import os

CHECKPOINT_PATH = "results/pipeline_a_checkpoint_8.json"

with open(CHECKPOINT_PATH, "r") as f:
    existing_results = json.load(f)

print("✓ Existing Pipeline A results loaded")
print("Existing completed tasks:", len(existing_results))

for r in existing_results:
    print(r["task_id"], "|", r["difficulty"])

✓ Existing Pipeline A results loaded
Existing completed tasks: 8
HumanEval/53 | easy
HumanEval/23 | easy
HumanEval/45 | easy
HumanEval/27 | easy
HumanEval/15 | easy
HumanEval/16 | easy
HumanEval/98 | easy
HumanEval/35 | easy


In [ ]:
# ============================================
# FIND REMAINING TASKS
# ============================================

completed_ids = {
    r["task_id"]
    for r in existing_results
}

remaining_tasks = [
    task
    for task in core_tasks
    if task["task_id"] not in completed_ids
]

print("=" * 60)
print("CONTINUATION STATUS")
print("=" * 60)

print("Already completed:", len(completed_ids))
print("Remaining:", len(remaining_tasks))

print("\nRemaining tasks:")

for i, task in enumerate(remaining_tasks, start=9):
    print(
        f"[{i}/30] "
        f"{task['task_id']} | "
        f"{task['difficulty']}"
    )

CONTINUATION STATUS
Already completed: 8
Remaining: 22

Remaining tasks:
[9/30] HumanEval/13 | easy
[10/30] HumanEval/83 | easy
[11/30] HumanEval/50 | medium
[12/30] HumanEval/58 | medium
[13/30] HumanEval/89 | medium
[14/30] HumanEval/26 | medium
[15/30] HumanEval/154 | medium
[16/30] HumanEval/140 | medium
[17/30] HumanEval/111 | medium
[18/30] HumanEval/93 | medium
[19/30] HumanEval/56 | medium
[20/30] HumanEval/31 | medium
[21/30] HumanEval/7 | hard
[22/30] HumanEval/25 | hard
[23/30] HumanEval/132 | hard
[24/30] HumanEval/54 | hard
[25/30] HumanEval/20 | hard
[26/30] HumanEval/6 | hard
[27/30] HumanEval/41 | hard
[28/30] HumanEval/126 | hard
[29/30] HumanEval/43 | hard
[30/30] HumanEval/99 | hard


In [ ]:
# ============================================
# PIPELINE A — CONTINUE FROM TASK 9
# ============================================

continuation_results = []

print("=" * 60)
print("PIPELINE A — CONTINUATION")
print("Already completed:", len(existing_results))
print("Remaining:", len(remaining_tasks))
print("Model:", MODEL)
print("Platform: Hugging Face")
print("=" * 60)

for position, task in enumerate(remaining_tasks, start=9):

    print(
        f"\n[{position}/30] "
        f"{task['task_id']} | "
        f"{task['difficulty']}"
    )

    try:
        result = single_agent_pipeline(task)

        continuation_results.append(result)

        print(
            f"✓ Completed | "
            f"Latency: {result['latency_sec']:.2f}s | "
            f"Tokens: {result['tokens']}"
        )

    except Exception as e:

        print(f"✗ ERROR: {e}")

        # Store failure separately so it cannot
        # accidentally be counted as a valid result.
        continuation_results.append({
            "task_id": task["task_id"],
            "difficulty": task["difficulty"],
            "generated_code": "",
            "latency_sec": None,
            "tokens": None,
            "token_source": "error",
            "error": str(e)
        })

print("\n" + "=" * 60)
print("CONTINUATION COMPLETE")
print("Records generated:", len(continuation_results))
print("=" * 60)

PIPELINE A — CONTINUATION
Already completed: 8
Remaining: 22
Model: meta-llama/Llama-3.1-8B-Instruct
Platform: Hugging Face

[9/30] HumanEval/13 | easy
✓ Completed | Latency: 1.51s | Tokens: 180

[10/30] HumanEval/83 | easy
✓ Completed | Latency: 1.82s | Tokens: 159

[11/30] HumanEval/50 | medium
✓ Completed | Latency: 2.09s | Tokens: 224

[12/30] HumanEval/58 | medium
✓ Completed | Latency: 2.14s | Tokens: 235

[13/30] HumanEval/89 | medium
✓ Completed | Latency: 2.47s | Tokens: 257

[14/30] HumanEval/26 | medium
✓ Completed | Latency: 1.62s | Tokens: 207

[15/30] HumanEval/154 | medium
✓ Completed | Latency: 1.19s | Tokens: 236

[16/30] HumanEval/140 | medium
✓ Completed | Latency: 1.32s | Tokens: 215

[17/30] HumanEval/111 | medium
✓ Completed | Latency: 5.61s | Tokens: 344

[18/30] HumanEval/93 | medium
✓ Completed | Latency: 3.67s | Tokens: 265

[19/30] HumanEval/56 | medium
✓ Completed | Latency: 2.32s | Tokens: 223

[20/30] HumanEval/31 | medium
✓ Completed | Latency: 2.33s | To

In [ ]:
# ============================================
# SAVE SECOND ACCOUNT CHECKPOINT
# ============================================

import json
import os

os.makedirs("results", exist_ok=True)

continuation_successful = [
    r for r in continuation_results
    if not r.get("error")
]

SECOND_CHECKPOINT = "results/pipeline_a_continuation_successful.json"

with open(SECOND_CHECKPOINT, "w") as f:
    json.dump(continuation_successful, f, indent=2)

print("=" * 60)
print("SECOND ACCOUNT CHECKPOINT")
print("=" * 60)
print("Successful:", len(continuation_successful))
print("Failed:", len(continuation_results) - len(continuation_successful))
print("Saved:", SECOND_CHECKPOINT)

SECOND ACCOUNT CHECKPOINT
Successful: 15
Failed: 7
Saved: results/pipeline_a_continuation_successful.json


In [ ]:
# ============================================
# VERIFY 23 VALID RESULTS
# ============================================

all_valid_so_far = existing_results + continuation_successful

ids = [r["task_id"] for r in all_valid_so_far]

print("Total valid results:", len(ids))
print("Unique task IDs:", len(set(ids)))

print("\nTasks:")
for i, task_id in enumerate(ids, start=1):
    print(f"{i:02d}. {task_id}")

Total valid results: 23
Unique task IDs: 23

Tasks:
01. HumanEval/53
02. HumanEval/23
03. HumanEval/45
04. HumanEval/27
05. HumanEval/15
06. HumanEval/16
07. HumanEval/98
08. HumanEval/35
09. HumanEval/13
10. HumanEval/83
11. HumanEval/50
12. HumanEval/58
13. HumanEval/89
14. HumanEval/26
15. HumanEval/154
16. HumanEval/140
17. HumanEval/111
18. HumanEval/93
19. HumanEval/56
20. HumanEval/31
21. HumanEval/7
22. HumanEval/25
23. HumanEval/132


In [ ]:
# ============================================
# PIPELINE A — RUN ONLY REMAINING 7 TASKS
# ============================================

import json
import os

# ------------------------------------------------
# 1. Load the existing 23 valid results
# ------------------------------------------------

CHECKPOINT_PATH = "results/pipeline_a_checkpoint_8.json"
CONTINUATION_PATH = "results/pipeline_a_continuation_successful.json"

with open(CHECKPOINT_PATH, "r") as f:
    first_8 = json.load(f)

with open(CONTINUATION_PATH, "r") as f:
    next_15 = json.load(f)

existing_results = first_8 + next_15

existing_ids = {
    r["task_id"]
    for r in existing_results
}

print("=" * 60)
print("EXISTING PIPELINE A RESULTS")
print("=" * 60)
print("Valid results:", len(existing_results))
print("Unique IDs:", len(existing_ids))

EXISTING PIPELINE A RESULTS
Valid results: 23
Unique IDs: 23


In [ ]:
# ------------------------------------------------
# 2. Identify the remaining 7 automatically
# ------------------------------------------------

remaining_tasks = [
    task
    for task in core_tasks
    if task["task_id"] not in existing_ids
]

print("\nRemaining tasks:", len(remaining_tasks))

for i, task in enumerate(remaining_tasks, start=24):
    print(
        f"[{i}/30] "
        f"{task['task_id']} | "
        f"{task['difficulty']}"
    )


Remaining tasks: 7
[24/30] HumanEval/54 | hard
[25/30] HumanEval/20 | hard
[26/30] HumanEval/6 | hard
[27/30] HumanEval/41 | hard
[28/30] HumanEval/126 | hard
[29/30] HumanEval/43 | hard
[30/30] HumanEval/99 | hard


In [ ]:
# ------------------------------------------------
# 3. Generate only the missing 7
# ------------------------------------------------

remaining_results = []

print("\n" + "=" * 60)
print("PIPELINE A — FINAL 7 TASKS")
print("=" * 60)

for position, task in enumerate(remaining_tasks, start=24):

    print(
        f"\n[{position}/30] "
        f"{task['task_id']} | "
        f"{task['difficulty']}"
    )

    try:
        result = single_agent_pipeline(task)

        remaining_results.append(result)

        print(
            f"✓ Completed | "
            f"Latency: {result['latency_sec']:.2f}s | "
            f"Tokens: {result['tokens']}"
        )

    except Exception as e:

        print(f"✗ ERROR: {e}")

        remaining_results.append({
            "task_id": task["task_id"],
            "difficulty": task["difficulty"],
            "generated_code": "",
            "latency_sec": None,
            "tokens": None,
            "token_source": "error",
            "error": str(e)
        })

print("\n" + "=" * 60)
print("FINAL 7 RUN COMPLETE")
print("Records:", len(remaining_results))
print("=" * 60)


PIPELINE A — FINAL 7 TASKS

[24/30] HumanEval/54 | hard
✓ Completed | Latency: 1.72s | Tokens: 240

[25/30] HumanEval/20 | hard
✓ Completed | Latency: 5.79s | Tokens: 346

[26/30] HumanEval/6 | hard
✓ Completed | Latency: 7.00s | Tokens: 282

[27/30] HumanEval/41 | hard
✓ Completed | Latency: 1.67s | Tokens: 248

[28/30] HumanEval/126 | hard
✓ Completed | Latency: 1.95s | Tokens: 346

[29/30] HumanEval/43 | hard
✓ Completed | Latency: 2.72s | Tokens: 270

[30/30] HumanEval/99 | hard
✓ Completed | Latency: 2.54s | Tokens: 268

FINAL 7 RUN COMPLETE
Records: 7


In [ ]:
# ------------------------------------------------
# 4. Save final-7 checkpoint
# ------------------------------------------------

os.makedirs("results", exist_ok=True)

FINAL_7_PATH = "results/pipeline_a_final_7.json"

with open(FINAL_7_PATH, "w") as f:
    json.dump(remaining_results, f, indent=2)

successful_7 = [
    r for r in remaining_results
    if not r.get("error")
]

print("✓ Final-7 checkpoint saved")
print("Successful:", len(successful_7))
print("Failed:", len(remaining_results) - len(successful_7))

✓ Final-7 checkpoint saved
Successful: 7
Failed: 0


In [ ]:
# ============================================
# SAFE MERGE: 23 + FINAL 7 = 30
# ============================================

all_results_by_id = {}

# Add existing 23
for result in existing_results:

    task_id = result["task_id"]

    if task_id in all_results_by_id:
        raise ValueError(
            f"Duplicate existing task: {task_id}"
        )

    all_results_by_id[task_id] = result


# Add the final 7
for result in successful_7:

    task_id = result["task_id"]

    if task_id in all_results_by_id:
        raise ValueError(
            f"DUPLICATE TASK DETECTED: {task_id}"
        )

    all_results_by_id[task_id] = result


pipeline_a_final = list(all_results_by_id.values())

print("=" * 60)
print("PIPELINE A — MERGED")
print("=" * 60)
print("Total valid results:", len(pipeline_a_final))
print("Unique task IDs:", len({
    r["task_id"] for r in pipeline_a_final
}))

PIPELINE A — MERGED
Total valid results: 30
Unique task IDs: 30


In [ ]:
# ============================================
# FINAL PIPELINE A VALIDATION
# ============================================

expected_ids = {
    task["task_id"]
    for task in core_tasks
}

final_ids = {
    result["task_id"]
    for result in pipeline_a_final
}

missing_ids = expected_ids - final_ids
extra_ids = final_ids - expected_ids

easy_count = sum(
    r["difficulty"] == "easy"
    for r in pipeline_a_final
)

medium_count = sum(
    r["difficulty"] == "medium"
    for r in pipeline_a_final
)

hard_count = sum(
    r["difficulty"] == "hard"
    for r in pipeline_a_final
)

print("=" * 60)
print("FINAL PIPELINE A VALIDATION")
print("=" * 60)

print("Expected tasks :", len(expected_ids))
print("Valid results  :", len(final_ids))
print("Missing        :", len(missing_ids))
print("Unexpected     :", len(extra_ids))
print("Easy           :", easy_count)
print("Medium         :", medium_count)
print("Hard           :", hard_count)

if missing_ids:
    print("\nMissing tasks:")
    for task_id in sorted(missing_ids):
        print(" ", task_id)

if extra_ids:
    print("\nUnexpected tasks:")
    for task_id in sorted(extra_ids):
        print(" ", task_id)

if (
    len(final_ids) == 30
    and len(pipeline_a_final) == 30
    and not missing_ids
    and not extra_ids
    and easy_count == 10
    and medium_count == 10
    and hard_count == 10
):
    print("\n✓ PIPELINE A COMPLETE: 30/30")
else:
    print("\n⚠ PIPELINE A NOT COMPLETE")

FINAL PIPELINE A VALIDATION
Expected tasks : 30
Valid results  : 30
Missing        : 0
Unexpected     : 0
Easy           : 10
Medium         : 10
Hard           : 10

✓ PIPELINE A COMPLETE: 30/30


In [ ]:
# ============================================
# SAVE OFFICIAL PIPELINE A DATASET
# ============================================

task_order = {
    task["task_id"]: i
    for i, task in enumerate(core_tasks)
}

pipeline_a_final.sort(
    key=lambda r: task_order[r["task_id"]]
)

FINAL_PATH = "results/pipeline_a_final_30.json"

with open(FINAL_PATH, "w") as f:
    json.dump(pipeline_a_final, f, indent=2)

print("=" * 60)
print("✓ OFFICIAL PIPELINE A DATASET SAVED")
print("=" * 60)
print("File:", FINAL_PATH)
print("Records:", len(pipeline_a_final))

✓ OFFICIAL PIPELINE A DATASET SAVED
File: results/pipeline_a_final_30.json
Records: 30


In [ ]:
from google.colab import files

files.download("results/pipeline_a_final_30.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>